# Synthetic Difference-in-Differences Demo (RU)

Демо покрывает базовый SDID, residualization + clustered SE, по-периодные эффекты, low-level helper-функции, wrapper и staggered adoption.

In [ ]:
import numpy as np
import pandas as pd

from cib_advanced_experiments import (
    SyntheticDifferenceInDifferences,
    fit_sdid,
    fit_staggered_sdid,
    fit_unit_weights,
    fit_time_weights,
    join_sdid_weights,
    residualize_outcome,
    calculate_regularization,
    calculate_time_regularization,
)

In [ ]:
df = pd.read_csv('notebooks/data/smoking.csv')
df['state'] = df['state'].astype(str)
df['date'] = pd.to_datetime(df['year'].astype(str) + '-01-01')

treated_state = df.loc[df['california'] == True, 'state'].iloc[0]
treatment_date = pd.Timestamp('1989-01-01')

did_data = df.copy()
did_data['treated'] = (did_data['state'] == treated_state).astype(int)
did_data['post'] = (did_data['date'] >= treatment_date).astype(int)
did_data[['state', 'date', 'cigsale', 'treated', 'post']].head()

In [ ]:
sdid = SyntheticDifferenceInDifferences(
    data=did_data,
    outcome_column='cigsale',
    unit_column='state',
    time_column='date',
    treatment_column='treated',
    post_column='post',
)
result = sdid.fit(alpha=0.05)

print('SDID ATT:', round(result.estimate, 4))
print('p-value:', round(result.pvalue, 4))
print('CI:', (round(result.ci_lower, 4), round(result.ci_upper, 4)))

In [ ]:
sdid.get_estimators_breakdown()

In [ ]:
result.unit_weights.head(10), result.time_weights.head(10)

In [ ]:
# Advanced fit: residualization + clustered SE
advanced_result = sdid.fit(
    alpha=0.05,
    covariate_columns=['retprice', 'age15to24'],
    cluster_by_unit=True,
    cluster_column='state',
)

print('Model outcome:', advanced_result.model_outcome_column)
print('Residualized:', advanced_result.residualized)
print('Clustered SE:', advanced_result.used_clustered_se)

In [ ]:
# Per-period SDID effects
period_effects = sdid.fit_effects_by_period(alpha=0.05)
period_effects.head()

In [ ]:
# Low-level wrapper compatible with fit_sdid payload
payload = fit_sdid(
    panel=did_data,
    outcome_col='cigsale',
    unit_column='state',
    time_column='date',
    treatment_column='treated',
    post_column='post',
    covariate_cols=['retprice', 'age15to24'],
)
{k: payload[k] for k in ['att', 'pvalue', 'model_outcome_col', 'used_clustered_se']}

In [ ]:
# Low-level weights helpers
unit_w = fit_unit_weights(
    panel=did_data,
    outcome_col='cigsale',
    unit_column='state',
    time_column='date',
    treatment_column='treated',
    post_column='post',
)
time_w = fit_time_weights(
    panel=did_data,
    outcome_col='cigsale',
    unit_column='state',
    time_column='date',
    treatment_column='treated',
    post_column='post',
)
joined_w = join_sdid_weights(
    panel=did_data,
    unit_weights=unit_w,
    time_weights=time_w,
    unit_column='state',
    time_column='date',
    treatment_column='treated',
    post_column='post',
    outcome_column='cigsale',
)
unit_w.head(), time_w.head(), joined_w[['unit_weight', 'time_weight', 'sdid_weight']].head()

In [ ]:
# Helper functions
resid_df, resid_col, resid_model = residualize_outcome(
    panel=did_data,
    outcome_col='cigsale',
    covariate_cols=['retprice', 'age15to24'],
    treatment_column='treated',
    post_column='post',
)
zeta = calculate_regularization(
    panel=did_data,
    outcome_col='cigsale',
    unit_column='state',
    time_column='date',
    treatment_column='treated',
    post_column='post',
)
zeta_time = calculate_time_regularization(
    panel=did_data,
    outcome_col='cigsale',
    unit_column='state',
    time_column='date',
    treatment_column='treated',
    post_column='post',
    ridge_scale=1e-6,
)
print('Residual column:', resid_col)
print('zeta:', zeta, 'zeta_time:', zeta_time)

In [ ]:
# Synthetic staggered-adoption demo
rng = np.random.default_rng(42)
dates = pd.date_range('2024-01-01', periods=12, freq='D')
never_treated = [f'c{i}' for i in range(8)]
cohort_a = ['t0', 't1']
cohort_b = ['t2', 't3']
starts = {'t0': pd.Timestamp('2024-01-06'), 't1': pd.Timestamp('2024-01-06'), 't2': pd.Timestamp('2024-01-09'), 't3': pd.Timestamp('2024-01-09')}

rows = []
for unit in never_treated + cohort_a + cohort_b:
    treated = int(unit.startswith('t'))
    unit_fe = rng.normal(0.0, 0.3)
    start = starts.get(unit)
    for idx, d in enumerate(dates):
        post = int(treated and d >= start)
        y = 15.0 + unit_fe + 0.1 * idx + 0.25 * treated + 0.2 * post + 2.0 * post + rng.normal(0.0, 0.08)
        rows.append({'unit': unit, 'date': d, 'treated': treated, 'post': post, 'y': y})
staggered_df = pd.DataFrame(rows)

staggered_res = fit_staggered_sdid(
    data=staggered_df,
    outcome_col='y',
    unit_column='unit',
    time_column='date',
    treatment_column='treated',
    post_column='post',
)

print('Aggregate staggered ATT:', round(staggered_res.aggregate_estimate, 4))
staggered_res.cohort_results

In [ ]:
sdid.plot_2x2()
sdid.plot_parallel_trends(treatment_start=treatment_date, pre_period_only=True)